In [1]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col

In [2]:
pyspark.__file__

'/home/daniel/anaconda3/lib/python3.13/site-packages/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('taxi_data_consolidated') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 20:22:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/green/')


In [5]:
df_yellow = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/yellow/')

In [6]:
# normalizing green columns: 
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('green')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'store_and_fwd_flag',
        'RatecodeID',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'ehail_fee',
        'improvement_surcharge',
        'total_amount',
        'payment_type',
        'trip_type',
        'congestion_surcharge',
        'taxi_type'
    )

In [7]:
# normalizing yellow columns: 
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('yellow')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'passenger_count',
        'trip_distance',
        'RatecodeID',
        'store_and_fwd_flag',
        'PULocationID',
        'DOLocationID',
        'payment_type',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'total_amount',
        'congestion_surcharge', 
        'taxi_type'
    )

In [8]:
commons_cols = []

for col_green in df_green.columns:
    for col_yellow in df_yellow.columns:
        
        if col_green == col_yellow: 
            commons_cols.append(col_green)
        
        else: 
            None

In [9]:
commons_cols

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'taxi_type']

In [10]:
df_green = df_green.select(commons_cols)

In [11]:
df_green.select(F.count('*')).show()

[Stage 2:=================================================>         (5 + 1) / 6]

+--------+
|count(1)|
+--------+
| 8348567|
+--------+



In [12]:
df_yellow = df_yellow.select(commons_cols)

In [13]:
df_yellow.select(F.count('*')).show()

[Stage 5:==================================================>      (30 + 4) / 34]

+---------+
| count(1)|
+---------+
|124048218|
+---------+



In [13]:
yellow_green_dataset_unioned = df_yellow.union(df_green)

In [15]:
yellow_green_dataset_unioned.select(F.count('*')).show()

[Stage 8:================================================>        (34 + 4) / 40]

+---------+
| count(1)|
+---------+
|132396785|
+---------+



In [14]:
df_yellow.createOrReplaceTempView('df_yellow_groupby')

In [15]:
df_yellow_groupby = spark.sql(
    """
    SELECT 
        PULocationID AS pickup_zone_ID, 
        DATE_TRUNC('MONTH', pickup_datetime) AS revenue_month,

        SUM(total_amount) AS Total_revenue_monthly,
        COUNT(PULocationID) AS Total_trips_monthly

    FROM 
        df_yellow_groupby
    GROUP BY 
        PULocationID,
        DATE_TRUNC('MONTH', pickup_datetime)
    ORDER BY 
        Total_revenue_monthly DESC NULLS LAST
    """
)

In [16]:
df_yellow_groupby.show()

[Stage 2:=======================================================> (33 + 1) / 34]

+--------------+-------------------+---------------------+-------------------+
|pickup_zone_ID|      revenue_month|Total_revenue_monthly|Total_trips_monthly|
+--------------+-------------------+---------------------+-------------------+
|           132|2019-05-01 00:00:00| 1.4015883699987967E7|             246361|
|           132|2019-10-01 00:00:00| 1.3570893849986957E7|             248324|
|           132|2019-06-01 00:00:00| 1.3383431569991782E7|             235445|
|           132|2019-09-01 00:00:00| 1.3221411419987496E7|             241466|
|           132|2019-04-01 00:00:00| 1.3190374339988995E7|             230420|
|           132|2019-03-01 00:00:00| 1.3179773499980763E7|             226740|
|           132|2019-07-01 00:00:00|  1.303985379999401E7|             237736|
|           132|2019-08-01 00:00:00| 1.2976801389993954E7|             240804|
|           132|2019-12-01 00:00:00|  1.239407688998855E7|             229793|
|           132|2019-11-01 00:00:00| 1.1969940419992

In [17]:
df_yellow_groupby = \
    df_yellow \
    .withColumn('revenue_month',F.date_trunc('month', df_yellow.pickup_datetime)) \
    .select(
        #Grouping columns: 
        df_yellow.PULocationID.alias('pickup_zone_ID'), 
        F.col('revenue_month'), 

        #Aggregations columns: 
        df_yellow.total_amount, 
        df_yellow.PULocationID
    ) \
    .filter(F.col('revenue_month') >= '2019-01-01 00:00:00') \
    .groupBy(
        F.col('pickup_zone_ID'), 
        F.col('revenue_month')
    ) \
    .agg(
        F.sum(df_yellow.total_amount).alias('Total_revenue_monthly_yellow'), 
        F.count(df_yellow.PULocationID).alias('Total_trips_monthly_yellow')
    ) \
    .orderBy(
        F.desc(F.col('Total_revenue_monthly_yellow'))
    )

In [18]:
df_yellow_groupby.show()

[Stage 5:=======================================================> (33 + 1) / 34]

+--------------+-------------------+----------------------------+--------------------------+
|pickup_zone_ID|      revenue_month|Total_revenue_monthly_yellow|Total_trips_monthly_yellow|
+--------------+-------------------+----------------------------+--------------------------+
|           132|2019-05-01 00:00:00|        1.4015883699987967E7|                    246361|
|           132|2019-10-01 00:00:00|        1.3570893849986957E7|                    248324|
|           132|2019-06-01 00:00:00|        1.3383431569991782E7|                    235445|
|           132|2019-09-01 00:00:00|        1.3221411419987496E7|                    241466|
|           132|2019-04-01 00:00:00|        1.3190374339988995E7|                    230420|
|           132|2019-03-01 00:00:00|        1.3179773499980763E7|                    226740|
|           132|2019-07-01 00:00:00|         1.303985379999401E7|                    237736|
|           132|2019-08-01 00:00:00|        1.2976801389993954E7|     

In [21]:
df_yellow_groupby \
    .repartition(20) \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/yellow', 
                   mode= 'overwrite'
                  )

In [19]:
df_green_groupby = \
    df_green \
    .withColumn('revenue_month',F.date_trunc('month', df_green.pickup_datetime)) \
    .select(
        #Grouping columns: 
        df_green.PULocationID.alias('pickup_zone_ID'), 
        F.col('revenue_month'), 

        #Aggregations columns: 
        df_green.total_amount, 
        df_green.PULocationID
    ) \
    .filter(F.col('revenue_month') >= '2019-01-01 00:00:00') \
    .groupBy(
        F.col('pickup_zone_ID'), 
        F.col('revenue_month')
    ) \
    .agg(
        F.sum(df_green.total_amount).alias('Total_revenue_monthly_green'), 
        F.count(df_green.PULocationID).alias('Total_trips_monthly_green')
    ) \
    .orderBy(
        F.desc(F.col('Total_revenue_monthly_green'))
    )

In [23]:
df_green_groupby.show(5)

[Stage 29:================================================>         (5 + 1) / 6]

+--------------+-------------------+---------------------------+-------------------------+
|pickup_zone_ID|      revenue_month|Total_revenue_monthly_green|Total_trips_monthly_green|
+--------------+-------------------+---------------------------+-------------------------+
|            74|2019-03-01 00:00:00|          574765.1499999098|                    42049|
|            74|2019-05-01 00:00:00|          567312.4299998907|                    40274|
|            74|2019-06-01 00:00:00|          561003.1399998823|                    39178|
|            74|2019-07-01 00:00:00|          553773.2699998681|                    39201|
|            74|2019-09-01 00:00:00|          553324.1199998844|                    37358|
+--------------+-------------------+---------------------------+-------------------------+
only showing top 5 rows


In [24]:
df_green_groupby \
    .repartition(20) \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/green', 
                   mode= 'overwrite'
                  )

In [25]:
# Using materialized results: 
df_green_groupby_imported = \
    spark.read.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/green')

df_yellow_groupby_imported = \
    spark.read.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/yellow')

In [26]:
df_joined = \
    df_green_groupby_imported \
    .join(
        df_yellow_groupby_imported, 
        on= ['pickup_zone_ID', 'revenue_month'], 
        how= 'outer'
    ) \
    .orderBy(
        F.asc(F.col('revenue_month'))
    )

In [27]:
df_joined.show(10)

[Stage 47:>                                                         (0 + 4) / 4]

+--------------+-------------------+---------------------------+-------------------------+----------------------------+--------------------------+
|pickup_zone_ID|      revenue_month|Total_revenue_monthly_green|Total_trips_monthly_green|Total_revenue_monthly_yellow|Total_trips_monthly_yellow|
+--------------+-------------------+---------------------------+-------------------------+----------------------------+--------------------------+
|             6|2019-01-01 00:00:00|                     631.54|                       15|          1259.6599999999999|                        33|
|            11|2019-01-01 00:00:00|         10491.629999999985|                      439|          3890.6799999999976|                       141|
|             1|2019-01-01 00:00:00|                     745.08|                        8|          41381.449999999924|                       446|
|             4|2019-01-01 00:00:00|         59.370000000000005|                        3|           197924.5000000129

In [28]:
df_joined \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/total', 
                   mode= 'overwrite'
                  )

In [29]:
df_total = \
    spark.read.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/total')

In [30]:
# !wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv

In [31]:
df_zones = \
    spark.read \
    .option('header','true') \
    .option('inferschema', 'true') \
    .csv('/home/daniel/notebooks/taxi_zone_lookup.csv')

In [32]:
df_zones \
    .write.parquet('/home/daniel/notebooks/taxi_zone_lookup.parquet', 
                  mode= 'overwrite')

In [33]:
df_zones = \
    spark.read \
    .option('header','true') \
    .option('inferschema', 'true') \
    .parquet('/home/daniel/notebooks/taxi_zone_lookup.parquet')

In [34]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [35]:
df_enriched = \
    df_total \
        .join(
            df_zones,
            df_total.pickup_zone_ID == df_zones.LocationID,
            how= 'left'
        )

In [36]:
df_enriched = \
    df_enriched.select(
        'pickup_zone_ID',
        'revenue_month',
        'Total_revenue_monthly_green',
        'Total_trips_monthly_green',
        'Total_revenue_monthly_yellow',
        'Total_trips_monthly_yellow',
        'Borough',
        'Zone',
        'service_zone'
    )

In [37]:
df_enriched \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/enriched', 
                   mode= 'overwrite')

## Resilient distribuited datasets (RDDs): 

In [20]:
#df_green before grouping to the see how it was done before when using RDDs
df_green.rdd

MapPartitionsRDD[25] at javaToPython at NativeMethodAccessorImpl.java:0

In [21]:
rdd = \
    df_green \
    .select(
        df_green.PULocationID, 
        df_green.pickup_datetime,
        df_green.total_amount,
    ) \
    .rdd

In [22]:
from datetime import datetime

In [23]:
start = datetime(year=2019, month=1, day=1)

In [24]:
def filter_outliers(row):
    return row.pickup_datetime >= start

In [25]:
rows = rdd.take(10)
row = rows[0]

In [26]:
def prepare_for_grouping(row):
    date = row.pickup_datetime.replace(minute= 0, second= 0, microsecond= 0)
    zone = row.PULocationID
    #composite key
    key = (date, zone)
        
    amount = row.total_amount
    count = 1
    #composite value
    value = (amount, count)

    return (key, value)

In [27]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

In [28]:
from collections import namedtuple

In [47]:
revenue_row = namedtuple('revenue_row', ['date', 'zone', 'revenue', 'count'])

In [48]:
def unwrap(row):
    return revenue_row(
        date = row[0][0], 
        zone = row[0][1], 
        revenue = row[1][0], 
        count = row[1][1]

    )

In [49]:
schema_rrd = types.StructType([
    types.StructField('date', types.TimestampType(), True), 
    types.StructField('zone', types.IntegerType(), True), 
    types.StructField('revenue', types.DoubleType(), True), 
    types.StructField('count', types.IntegerType(), True)
])

In [50]:
df_results_rrd = \
    rdd \
        .filter(filter_outliers) \
        .map(prepare_for_grouping) \
        .reduceByKey(calculate_revenue) \
        .map(unwrap) \
        .toDF(schema_rrd)

In [51]:
df_results_rrd.show(5)

[Stage 72:================================================>         (5 + 1) / 6]

+-------------------+----+------------------+-----+
|               date|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2019-01-30 00:00:00|  82| 295.2600000000001|   28|
|2019-01-19 23:00:00|  41| 613.0499999999998|   48|
|2019-01-09 19:00:00| 260| 441.3900000000002|   37|
|2019-01-28 10:00:00| 179|             76.66|    8|
|2019-01-03 11:00:00|  42|454.51000000000016|   36|
+-------------------+----+------------------+-----+
only showing top 5 rows


In [52]:
df_results_rrd \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/rdd/', 
                   mode= 'overwrite')

Traceback (most recent call last):
  File "/home/daniel/spark/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/home/daniel/spark/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe
                                                                                

In [53]:
!tree -lh taxi_data_reporting

[4.0K]  taxi_data_reporting
└── [4.0K]  revenue
    ├── [4.0K]  enriched
    │   ├── [   0]  _SUCCESS
    │   └── [171K]  part-00000-7c850a09-11bc-41af-9c89-5ab85c7fa50c-c000.snappy.parquet
    ├── [4.0K]  green
    │   ├── [   0]  _SUCCESS
    │   ├── [7.5K]  part-00000-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.3K]  part-00001-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.4K]  part-00002-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.3K]  part-00003-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.3K]  part-00004-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.4K]  part-00005-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.4K]  part-00006-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.4K]  part-00007-ab2e79ff-3654-4127-af70-88d465719489-c000.snappy.parquet
    │   ├── [7.2K]  part-00008-ab2e79ff-3654-4127-af70-

## RDDs: mapPartitions functions

In [11]:
duration_rdd = \
    df_green \
        .select(
            df_green.VendorID, 
            df_green.pickup_datetime,
            df_green.PULocationID, 
            df_green.DOLocationID,
            df_green.trip_distance
        ) \
        .rdd

In [12]:
rows = duration_rdd.take(10)

In [13]:
def model_predict(pandas_df): 
    y_pred = pandas_df.trip_distance * 5

    return y_pred

In [14]:
def apply_model_batch(rows):
    pandas_df = pd.DataFrame(
        rows, 
        columns = ['VendorID', 
                   'pickup_datetime', 
                   'PULocationID', 
                   'DOLocationID', 
                   'trip_distance'
                  ]
    )
    predictions = model_predict(pandas_df)
    pandas_df['predicted_duration'] = predictions
    #for each row it will have a tuple
    for row in pandas_df.itertuples(): 
        yield row

In [16]:
df_prediction = duration_rdd \
    .mapPartitions(apply_model_batch) \
    .toDF() \
    .drop('Index')

In [17]:
df_prediction.show()

[Stage 5:>                                                          (0 + 1) / 1]

+--------+---------------+------------+------------+-------------+------------------+
|VendorID|pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+--------+---------------+------------+------------+-------------+------------------+
|       2|             {}|          82|         160|         2.91|             14.55|
|       2|             {}|          41|          74|         1.08|               5.4|
|       2|             {}|          82|         129|         1.24|               6.2|
|       2|             {}|         130|          10|          2.5|              12.5|
|       2|             {}|         119|         226|         8.72|              43.6|
|       2|             {}|          75|         151|         1.58|               7.9|
|       2|             {}|          41|          42|         0.82|               4.1|
|       2|             {}|         183|          81|         3.26|16.299999999999997|
|       2|             {}|          72|          61|  

Traceback (most recent call last):                                              
  File "/home/daniel/spark/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/home/daniel/spark/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe
